# ACS Model Variable Extraction

**Run this once.** Loads only the 9 columns needed by the MRP model from all four
ACS 5-year PUMS files (16 M rows), applies all recodes, filters to adults 18+,
and saves `data/acs_adults_model_vars.parquet`.

All other notebooks load from that parquet (seconds vs. minutes).

### Variables extracted (from model PDF)
| ACS column | Model variable | Notes |
|---|---|---|
| `PWGTP` | person weight | poststrat denominator |
| `STATE` | state FIPS | geographic intercept |
| `DIVISION` | region9 | Census division (1–9) |
| `PUMA` | puma_code | needed for county-level crosswalk |
| `SEX` → `gender` | Male / Female | individual-level RE |
| `RAC1P` + `HISP` → `race4` | White / Black / Hispanic / Other | individual-level RE |
| `SCHL` → `educ_category` | 1–4 | individual-level RE |
| `AGEP` | age filter | adults 18+ only |

In [ ]:
import pandas as pd
import numpy as np

ACS_DIR  = "/Users/carmenk/Documents/CSS/Capstone/acs2024_csv_5year/"
OUT_PATH = "/Users/carmenk/Documents/CSS/Capstone/data/acs_adults_model_vars.parquet"

# Only the 9 columns the model needs
COLS = ["PWGTP", "STATE", "PUMA", "DIVISION", "SEX", "RAC1P", "HISP", "SCHL", "AGEP"]

parts = []
for suffix in ["a", "b", "c", "d"]:
    chunk = pd.read_csv(f"{ACS_DIR}psam_pus{suffix}.csv",
                        usecols=COLS, low_memory=False)
    parts.append(chunk)
    print(f"  psam_pus{suffix}.csv → {len(chunk):,} rows")

pums = pd.concat(parts, ignore_index=True)
print(f"\nTotal: {len(pums):,}")

In [ ]:
# ── Recodes ───────────────────────────────────────────────────────────────────

# Adults only
pums = pums[pums["AGEP"] >= 18].copy()
print(f"Adults 18+: {len(pums):,}")

# gender
pums["gender"] = pums["SEX"].map({1: "Male", 2: "Female"})

# race4  — ethnicity (HISP) takes precedence over race (RAC1P)
hisp = pd.to_numeric(pums["HISP"], errors="coerce")
pums["race4"] = np.select(
    [hisp > 1,
     (hisp <= 1) & (pums["RAC1P"] == 1),
     (hisp <= 1) & (pums["RAC1P"] == 2)],
    ["Hispanic", "White", "Black"],
    default="Other"
)

# educ_category: (0,15]=1 LessHS | (15,17]=2 HS | (17,20]=3 SomeCol | (20,24]=4 BA+
pums["educ_category"] = pd.cut(
    pd.to_numeric(pums["SCHL"], errors="coerce"),
    bins=[0, 15, 17, 20, 24], labels=[1, 2, 3, 4]
).astype("Int8")

# region9 label from DIVISION
division_map = {
    1: "New England",    2: "Mid-Atlantic",      3: "E. North Central",
    4: "W. North Central", 5: "South Atlantic",  6: "E. South Central",
    7: "W. South Central", 8: "Mountain",        9: "Pacific",
}
pums["region9"] = pums["DIVISION"].map(division_map)

# zero-pad geographic keys for crosswalk
pums["state_fips"] = pums["STATE"].astype(str).str.zfill(2)
pums["puma_code"]  = pums["PUMA"].astype(str).str.zfill(5)

# Drop raw columns no longer needed
pums = pums.drop(columns=["SEX", "RAC1P", "HISP", "SCHL", "AGEP", "PUMA"])

print(f"\nFinal columns: {pums.columns.tolist()}")
print(pums.head(5).to_string())

In [ ]:
# ── Save to Parquet (fast reload in future notebooks) ─────────────────────────
pums.to_parquet(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH}")
print(f"Rows: {len(pums):,} | Columns: {pums.shape[1]}")

# Verify round-trip
check = pd.read_parquet(OUT_PATH)
print(f"Verified read-back: {len(check):,} rows")